# Lab 5: Linear PID Control and Linear Interpolation

Goal: drive the robot as fast as possible toward a wall and stop exactly **1 ft (304 mm)** away using TOF sensor feedback.

Experiments:
1. **P control** — baseline, tune KP
2. **PD control** — add derivative to reduce overshoot
3. **PID control + wind-up clamp** — add integral for steady-state accuracy
4. **Extrapolation validation** — decouple PID loop rate from TOF rate

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import importlib

from ble import get_ble_controller
from base_ble import LOG
import cmd_types
importlib.reload(cmd_types)
from cmd_types import CMD

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams['font.size'] = 12

SETPOINT_MM = 304  # 1 ft

# ── Motor PWM constants (must match Arduino firmware) ─────────────────────────
# Firmware: mapped_PWM = DEADBAND_MIN + (|output| / FW_PWM_MAX) * (FW_PWM_MAX - DEADBAND_MIN)
# To cap robot speed at MOTOR_PWM_MAX, we need:
#   |PID output| ≤ (MOTOR_PWM_MAX - DEADBAND_MIN) / (FW_PWM_MAX - DEADBAND_MIN) * FW_PWM_MAX
FW_PWM_MAX    = 200   # Arduino #define PWM_MAX
DEADBAND_MIN  = 40    # Arduino #define DEADBAND_MIN
MOTOR_PWM_MAX = 60    # desired max motor PWM (lab4 confirmed 60 is fast enough)

# Max allowed PID output to achieve MOTOR_PWM_MAX:
_MAX_OUTPUT = (MOTOR_PWM_MAX - DEADBAND_MIN) / (FW_PWM_MAX - DEADBAND_MIN) * FW_PWM_MAX
# _MAX_OUTPUT ≈ 25  →  KP = _MAX_OUTPUT / max_expected_error
print(f"Max PID output for {MOTOR_PWM_MAX} PWM cap: {_MAX_OUTPUT:.1f}")

## 2. Connect to Artemis via BLE

In [ ]:
ble = get_ble_controller()
ble.connect()
print("Connected!")


def ble_reconnect():
    """断线重连，BLE 进入坏状态时调用。"""
    global ble
    print("Reconnecting BLE ...")
    try:
        ble.disconnect()
    except Exception:
        pass
    time.sleep(1.0)
    ble = get_ble_controller()
    ble.connect()
    print("Reconnected!")

---
## 3. Prelab — BLE Helper Functions

### Design
1. `pid_start()` — sends `PID_START`; Artemis begins PID, stores data, hard-stops after 10 s
2. `pid_stop()` — emergency stop via `PID_STOP`
3. `set_pid_gains(kp, ki, kd, setpoint)` — tune gains over BLE without reflashing
4. `run_pid_experiment()` — orchestrates start → wait → stop → retrieve
5. `parse_pid_data()` — splits notification stream into TOF DataFrame and PID-loop DataFrame

Data format from Artemis:
```
TOF|{dist_mm}|{extrap_flag}|{time_ms}   # extrap_flag: 0=real, 1=extrapolated
PID|{error_mm}|{motor_pwm}|{time_ms}
PID_END|{tof_count}|{pid_count}
```

In [ ]:
# ── Low-level BLE wrappers ────────────────────────────────────────────────────

def pid_start():
    """Tell Artemis to start a PID run."""
    ble.send_command(CMD.PID_START, "")
    time.sleep(0.1)
    return ble.receive_string(ble.uuid['RX_STRING'])

def pid_stop():
    """Emergency-stop PID (also called automatically after timeout)."""
    ble.send_command(CMD.PID_STOP, "")
    time.sleep(0.1)
    return ble.receive_string(ble.uuid['RX_STRING'])

def set_pid_gains(kp, ki, kd, setpoint=304):
    """Push PID gains to Artemis without reflashing."""
    ble.send_command(CMD.SET_PID_GAINS, f"{kp}|{ki}|{kd}|{setpoint}")
    time.sleep(0.15)
    resp = ble.receive_string(ble.uuid['RX_STRING'])
    print(f"Gains set → {resp}")
    return resp

print("Low-level BLE helpers defined.")

In [ ]:
# ── Notification-based data collection ───────────────────────────────────────

_pid_buf  = []
_pid_done = False

def _pid_notify_handler(uuid, bytearray_data):
    global _pid_buf, _pid_done
    try:
        line = ble.bytearray_to_string(bytearray_data).strip()
        _pid_buf.append(line)
        if line.startswith('PID_END'):
            _pid_done = True
    except Exception as ex:
        print(f"Handler error: {ex}")


def parse_pid_data(buf):
    tof_rows, pid_rows = [], []
    for line in buf:
        if line.startswith('TOF|'):
            parts = line.split('|')
            if len(parts) == 4:
                tof_rows.append({
                    'dist_mm': int(parts[1]),
                    'extrap':  bool(int(parts[2])),
                    'time_ms': int(parts[3]),
                })
        elif line.startswith('PID|'):
            parts = line.split('|')
            if len(parts) == 4:
                pid_rows.append({
                    'error_mm':  int(parts[1]),
                    'motor_pwm': int(parts[2]),
                    'time_ms':   int(parts[3]),
                })

    tof_df = pd.DataFrame(tof_rows)
    pid_df = pd.DataFrame(pid_rows)

    mins = []
    if len(tof_df): mins.append(tof_df['time_ms'].min())
    if len(pid_df): mins.append(pid_df['time_ms'].min())
    if mins:
        t0 = min(mins)
        if len(tof_df): tof_df['time_s'] = (tof_df['time_ms'] - t0) / 1000.0
        if len(pid_df): pid_df['time_s'] = (pid_df['time_ms'] - t0) / 1000.0

    return tof_df, pid_df


def run_pid_experiment(run_duration_s=10.0, label=""):
    global _pid_buf, _pid_done
    _pid_buf  = []
    _pid_done = False

    # 清理残留 notification subscription
    try:
        ble.stop_notify(ble.uuid['RX_STRING'])
    except Exception:
        pass

    ble.start_notify(ble.uuid['RX_STRING'], _pid_notify_handler)
    time.sleep(0.05)

    print(f"  [{label}] Sending PID_START ...")
    ble.send_command(CMD.PID_START, "")
    time.sleep(run_duration_s + 1.0)

    ble.send_command(CMD.PID_STOP, "")
    time.sleep(0.2)

    print(f"  [{label}] Retrieving data ...")
    _pid_done = False
    ble.send_command(CMD.GET_PID_DATA, "")

    t0 = time.time()
    while not _pid_done and (time.time() - t0 < 30.0):
        time.sleep(0.1)

    ble.stop_notify(ble.uuid['RX_STRING'])

    tof_df, pid_df = parse_pid_data(_pid_buf)
    print(f"  [{label}] Got {len(tof_df)} TOF samples, {len(pid_df)} PID samples")

    if len(tof_df) > 1:
        real = tof_df[~tof_df['extrap']]
        if len(real) > 1:
            tof_interval = real['time_ms'].diff().dropna().mean()
            print(f"  [{label}] Avg TOF interval: {tof_interval:.1f} ms  ({1000/tof_interval:.1f} Hz)")
    if len(pid_df) > 1:
        pid_interval = pid_df['time_ms'].diff().dropna().mean()
        print(f"  [{label}] Avg PID loop interval: {pid_interval:.1f} ms  ({1000/pid_interval:.1f} Hz)")

    return tof_df, pid_df


print("Data collection helpers defined.")

In [ ]:
# -- Plotting helper ---------------------------------------------------------

def _estimate_motor_from_tof(tof_df, kp, ki=0.0, kd=0.0):
    """Estimate motor PWM from TOF data using P+D terms (best effort when pid_df is empty)."""
    s = tof_df.copy().sort_values('time_ms').reset_index(drop=True)
    real = s[~s['extrap']]
    if len(real) == 0:
        real = s
    real = real.reset_index(drop=True)
    err = (real['dist_mm'] - SETPOINT_MM).astype(float)
    out = kp * err
    if kd > 0 and len(real) > 1:
        dt_s = real['time_ms'].diff() / 1000.0
        deriv = err.diff() / dt_s.replace(0, np.nan)
        out = out + kd * deriv.fillna(0)
    sign = np.sign(out)
    mag = out.abs().clip(0, FW_PWM_MAX)
    motor = sign * np.where(mag > 0,
                            DEADBAND_MIN + (mag / FW_PWM_MAX) * (FW_PWM_MAX - DEADBAND_MIN),
                            0.0)
    return real['time_s'], pd.Series(motor, index=real.index)


def plot_pid_results(tof_df, pid_df, title='PID Control', save=True, kp=0.0, ki=0.0, kd=0.0):
    """
    Three-panel figure:
      Top    - TOF distance vs time (real + extrapolated)
      Middle - Error vs time  (from pid_df if available, else derived from tof_df)
      Bottom - Motor PWM vs time (from pid_df; estimated from TOF+gains if unavailable)
    """
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

    # -- TOF distance ---------------------------------------------------------
    ax = axes[0]
    if len(tof_df):
        real   = tof_df[~tof_df['extrap']]
        extrap = tof_df[ tof_df['extrap']]
        ax.plot(real['time_s'],   real['dist_mm'],   'b.',  ms=5, label='TOF real')
        if len(extrap):
            ax.plot(extrap['time_s'], extrap['dist_mm'], 'c.', ms=2,
                    alpha=0.6, label='TOF extrapolated')
    ax.axhline(SETPOINT_MM, color='r', linestyle='--', lw=1.5, label=f'Setpoint {SETPOINT_MM} mm')
    ax.set_ylabel('Distance (mm)')
    ax.set_title(f'{title} - TOF Distance vs Time')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    # -- Error ----------------------------------------------------------------
    ax = axes[1]
    if len(pid_df):
        ax.plot(pid_df['time_s'], pid_df['error_mm'], 'g-', lw=1.5)
    elif len(tof_df):
        sc = tof_df.copy()
        sc['error_mm'] = sc['dist_mm'] - SETPOINT_MM
        real_src = sc[~sc['extrap']]
        plot_src = real_src if len(real_src) > 0 else sc  # fallback: all data
        ax.plot(plot_src['time_s'], plot_src['error_mm'], 'g-', lw=1.5,
                label='Error (from TOF)')
        ax.legend(fontsize=9)
    ax.axhline(0, color='r', linestyle='--', lw=1, alpha=0.7)
    ax.set_ylabel('Error (mm)')
    ax.set_title('Error vs Time')
    ax.grid(alpha=0.3)

    # -- Motor PWM ------------------------------------------------------------
    ax = axes[2]
    if len(pid_df):
        ax.plot(pid_df['time_s'], pid_df['motor_pwm'], 'm-', lw=1.5)
    elif len(tof_df) and kp > 0:
        t_e, pwm_e = _estimate_motor_from_tof(tof_df, kp=kp, ki=ki, kd=kd)
        ax.plot(t_e, pwm_e, 'm--', lw=1.5, label='Motor PWM (estimated)')
        ax.legend(fontsize=9)
    else:
        ax.text(0.5, 0.5, 'Motor PWM not logged\n(no PID| lines received)',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=11, color='gray')
    ax.axhline(0,    color='k', linestyle='-',  lw=0.5)
    ax.axhline( 40,  color='orange', linestyle=':', lw=1, alpha=0.7, label='Deadband +-40')
    ax.axhline(-40,  color='orange', linestyle=':', lw=1, alpha=0.7)
    ax.set_ylabel('Motor PWM')
    ax.set_xlabel('Time (s)')
    ax.set_title('Motor Output vs Time')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save:
        fname = f'lab5_{title.replace(" ", "_").lower()}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'Saved -> {fname}')
    plt.show()
    return fig


print('Plot helper defined.')

---
## 4. Experiment 1 — P Control

**Theory:** KP = 166 / 2000 ≈ 0.083 (max PWM at 2 m from target).  
Start there and tune down until the robot can stop without crashing.

Expected behaviour: car oscillates around setpoint before settling (P-only has no mechanism to kill overshoot).

In [ ]:
# ── 先用 PING 确认连接，再读一次即时 TOF 距离 ────────────────────────────────
ble.send_command(CMD.PING, "")
print("PING:", ble.receive_string(ble.uuid['RX_STRING']))

ble.send_command(CMD.GET_TOF_DATA, "")
time.sleep(0.5)
tof_now = ble.receive_string(ble.uuid['RX_STRING'])
print("TOF reading:", tof_now)
print()
print("⚠ 如果读数远小于实际距离（比如应该是 2m 却显示 300-400mm），")
print("  说明 TOF 传感器朝向不对，请调整机器人朝向后再跑 PID。")

# ── Set P-only gains (KI=0, KD=0) ────────────────────────────────────────────
# KP = _MAX_OUTPUT / max_expected_error = 25 / 1700 ≈ 0.015
# At 1700 mm error → output ≈ 25 → mapped PWM = 40 + (25/200)*160 ≈ 60
KP = 0.015
set_pid_gains(kp=KP, ki=0.0, kd=0.0, setpoint=SETPOINT_MM)

In [ ]:
# ── Run P-only experiment ─────────────────────────────────────────────────────
try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception:
    pass
input("Place robot 2–4 m from wall, then press Enter to start ...")
tof_p, pid_p = run_pid_experiment(run_duration_s=10.0, label="P-only")

# ── Debug: 前5次真实 TOF 读数 ─────────────────────────────────────────────────
print("\n===== DEBUG: P-only =====")
real_p = tof_p[~tof_p['extrap']] if len(tof_p) else pd.DataFrame()
if len(real_p):
    print(f"前5次真实 TOF 读数 (setpoint={SETPOINT_MM}mm):")
    for _, row in real_p.head(5).iterrows():
        err = row['dist_mm'] - SETPOINT_MM
        print(f"  t={row['time_s']:.3f}s  dist={row['dist_mm']}mm  error={err:+.0f}mm")
else:
    print("⚠ 没有收到任何 TOF 数据！")
print(f"TOF总样本: {len(tof_p)}  PID总样本: {len(pid_p)}")
print("=========================")

In [ ]:
plot_pid_results(tof_p, pid_p, title='P Control', kp=KP)

# Max speed estimate from TOF data
if len(tof_p) > 1:
    real_p = tof_p[~tof_p['extrap']].copy()
    if len(real_p) > 1:
        real_p = real_p.sort_values('time_ms')
        speeds = -real_p['dist_mm'].diff() / real_p['time_ms'].diff()  # mm/ms = m/s
        max_speed = speeds.max()
        print(f'Estimated max approach speed: {max_speed:.3f} m/s  ({max_speed*3.281:.2f} ft/s)')

In [ ]:
# -- P control: 3 trials at max PWM ~ 40 / 80 / 120 -------------------------
PWM_TARGETS  = [40,     80,     120   ]
KP_P_TRIALS  = [0.005,  0.029,  0.059 ]   # ~47, ~80, ~120 mapped PWM

p_trials = []
for pwm_t, kp_t in zip(PWM_TARGETS, KP_P_TRIALS):
    set_pid_gains(kp=kp_t, ki=0.0, kd=0.0, setpoint=SETPOINT_MM)
    input(f'[P PWM~{pwm_t}] Place robot 2-4 m from wall, press Enter ...')
    tof_t, pid_t = run_pid_experiment(run_duration_s=10.0, label=f'P PWM~{pwm_t}')
    p_trials.append((tof_t, pid_t, pwm_t))

set_pid_gains(kp=KP, ki=0.0, kd=0.0, setpoint=SETPOINT_MM)

# -- Plot --------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'darkorange', 'forestgreen']

ax = axes[0]
for (tof_t, _, pwm_t), color in zip(p_trials, colors):
    real = tof_t[~tof_t['extrap']]
    ax.plot(real['time_s'], real['dist_mm'], color=color, lw=1.5, label=f'max PWM~{pwm_t}')
ax.axhline(SETPOINT_MM, color='r', linestyle='--', lw=1.5, label=f'Setpoint {SETPOINT_MM}mm')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Distance (mm)')
ax.set_title('P Control - TOF Distance')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
for (tof_t, pid_t, pwm_t), kp_t, color in zip(p_trials, KP_P_TRIALS, colors):
    if len(pid_t):
        ax.plot(pid_t['time_s'], pid_t['motor_pwm'], color=color, lw=1.5, label=f'max PWM~{pwm_t}')
    elif len(tof_t):
        t_e, pwm_e = _estimate_motor_from_tof(tof_t, kp=kp_t)
        ax.plot(t_e, pwm_e, color=color, lw=1.5, linestyle='--', label=f'max PWM~{pwm_t} (est.)')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Motor PWM')
ax.set_title('P Control - Motor Output')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('P Control: max PWM 40 / 80 / 120', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lab5_p_pwm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Experiment 2 — PD Control

Adding the derivative term damps the control signal when the error is changing quickly  
(i.e., while the robot is approaching fast), reducing overshoot.  
The derivative uses an LPF with α = 0.9 to suppress sensor noise.

**Tuning starting point:** KD ≈ KP / 3 ≈ 0.016

In [ ]:
KP = 0.015
KD = 0.004   # KD/KP ≈ 0.27, same ratio as original (0.016/0.055)
set_pid_gains(kp=KP, ki=0.0, kd=KD, setpoint=SETPOINT_MM)

In [ ]:
try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception:
    pass
input("Place robot 2–4 m from wall, then press Enter to start ...")
tof_pd, pid_pd = run_pid_experiment(run_duration_s=10.0, label="PD")

# ── Debug: 前5次真实 TOF 读数 ─────────────────────────────────────────────────
print("\n===== DEBUG: PD =====")
real_pd = tof_pd[~tof_pd['extrap']] if len(tof_pd) else pd.DataFrame()
if len(real_pd):
    print(f"前5次真实 TOF 读数 (setpoint={SETPOINT_MM}mm):")
    for _, row in real_pd.head(5).iterrows():
        err = row['dist_mm'] - SETPOINT_MM
        print(f"  t={row['time_s']:.3f}s  dist={row['dist_mm']}mm  error={err:+.0f}mm")
else:
    print("⚠ 没有收到任何 TOF 数据！")
print(f"TOF总样本: {len(tof_pd)}  PID总样本: {len(pid_pd)}")
print("=====================")

In [ ]:
plot_pid_results(tof_pd, pid_pd, title='PD Control', kp=KP, kd=KD)

In [ ]:
# ── P vs PD comparison ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (tof_x, pid_x), label, color in [
    (axes[0], (tof_p,  pid_p),  'P only',  'steelblue'),
    (axes[1], (tof_pd, pid_pd), 'PD',       'darkorange'),
]:
    real = tof_x[~tof_x['extrap']]
    ax.plot(real['time_s'], real['dist_mm'], color=color, lw=1.5)
    ax.axhline(SETPOINT_MM, color='r', linestyle='--', lw=1.5, label='Setpoint')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Distance (mm)')
    ax.set_title(f'{label} — TOF vs Time')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('P vs PD: TOF Distance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lab5_p_vs_pd_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -- PD control: 3 trials at max PWM ~ 40 / 80 / 120 ------------------------
KD_KP_RATIO  = 0.016 / 0.055
KP_PD_TRIALS = [0.005,  0.029,  0.059 ]
KD_PD_TRIALS = [round(kp * KD_KP_RATIO, 4) for kp in KP_PD_TRIALS]

pd_trials = []
for pwm_t, kp_t, kd_t in zip(PWM_TARGETS, KP_PD_TRIALS, KD_PD_TRIALS):
    set_pid_gains(kp=kp_t, ki=0.0, kd=kd_t, setpoint=SETPOINT_MM)
    input(f'[PD PWM~{pwm_t}] Place robot 2-4 m from wall, press Enter ...')
    tof_t, pid_t = run_pid_experiment(run_duration_s=10.0, label=f'PD PWM~{pwm_t}')
    pd_trials.append((tof_t, pid_t, pwm_t))

set_pid_gains(kp=KP, ki=0.0, kd=KD, setpoint=SETPOINT_MM)

# -- Plot --------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'darkorange', 'forestgreen']

ax = axes[0]
for (tof_t, _, pwm_t), color in zip(pd_trials, colors):
    real = tof_t[~tof_t['extrap']]
    ax.plot(real['time_s'], real['dist_mm'], color=color, lw=1.5, label=f'max PWM~{pwm_t}')
ax.axhline(SETPOINT_MM, color='r', linestyle='--', lw=1.5, label=f'Setpoint {SETPOINT_MM}mm')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Distance (mm)')
ax.set_title('PD Control - TOF Distance')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
for (tof_t, pid_t, pwm_t), kp_t, kd_t, color in zip(pd_trials, KP_PD_TRIALS, KD_PD_TRIALS, colors):
    if len(pid_t):
        ax.plot(pid_t['time_s'], pid_t['motor_pwm'], color=color, lw=1.5, label=f'max PWM~{pwm_t}')
    elif len(tof_t):
        t_e, pwm_e = _estimate_motor_from_tof(tof_t, kp=kp_t, kd=kd_t)
        ax.plot(t_e, pwm_e, color=color, lw=1.5, linestyle='--', label=f'max PWM~{pwm_t} (est.)')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Motor PWM')
ax.set_title('PD Control - Motor Output')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('PD Control: max PWM 40 / 80 / 120', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lab5_pd_pwm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Experiment 3 — PID Control with Wind-up Protection

The integral accumulates over time to eliminate steady-state error (e.g., the robot  
sitting a few mm off target because the proportional term alone isn't enough to move it).

### Wind-up problem
If the robot is held in hand for several seconds before release, the integral builds  
to a huge value. When released, the robot runs at max PWM and cannot decelerate in time.  
**Clamp:** the integral is clamped to ±1000 (mm·s units) inside `handle_pid()` on the Artemis.

**Tuning:** KI ≈ 0.003 (small — the integral is a gentle correction, not the main force).

In [ ]:
KP = 0.015
KI = 0.001   # KI/KP ≈ 0.067, same ratio as original (0.003/0.055)
KD = 0.004
set_pid_gains(kp=KP, ki=KI, kd=KD, setpoint=SETPOINT_MM)

In [ ]:
try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception:
    pass
input("Place robot 2–4 m from wall, then press Enter to start ...")
tof_pid, pid_pid = run_pid_experiment(run_duration_s=10.0, label="PID")

# ── Debug: 前5次真实 TOF 读数 ─────────────────────────────────────────────────
print("\n===== DEBUG: PID =====")
real_pid = tof_pid[~tof_pid['extrap']] if len(tof_pid) else pd.DataFrame()
if len(real_pid):
    print(f"前5次真实 TOF 读数 (setpoint={SETPOINT_MM}mm):")
    for _, row in real_pid.head(5).iterrows():
        err = row['dist_mm'] - SETPOINT_MM
        print(f"  t={row['time_s']:.3f}s  dist={row['dist_mm']}mm  error={err:+.0f}mm")
else:
    print("⚠ 没有收到任何 TOF 数据！")
print(f"TOF总样本: {len(tof_pid)}  PID总样本: {len(pid_pid)}")
print("======================")

In [ ]:
plot_pid_results(tof_pid, pid_pid, title='PID Control', kp=KP, ki=KI, kd=KD)

### 6a. Wind-up Demonstration

Hold the robot in hand for ~5 s before releasing so the integral builds up.  
Run **without** clamp (temporarily set clamp limit very high by using large KI)  
to show the runaway behaviour, then repeat with the real KI to show the fix.

In [ ]:
# ── Wind-up scenario: hold car in hand for 5 s, then release ─────────────────
# Use a much larger KI to saturate the integral quickly
print("Setting large KI to demonstrate wind-up ...")
set_pid_gains(kp=KP, ki=0.05, kd=0.0, setpoint=SETPOINT_MM)  # high KI, no D

print("\n  → HOLD THE CAR in your hand for the first 5 s, then place 2–4 m from wall and release.")
input("Press Enter when ready ...")
try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception:
    pass
tof_wu, pid_wu = run_pid_experiment(run_duration_s=10.0, label="windup demo")

In [ ]:
# ── With wind-up clamp (normal KI, Artemis clamps at ±1000) ──────────────────
set_pid_gains(kp=KP, ki=KI, kd=KD, setpoint=SETPOINT_MM)

print("  → HOLD THE CAR in your hand for the first 5 s, then place 2–4 m from wall and release.")
input("Press Enter when ready ...")
try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception:
    pass
tof_wf, pid_wf = run_pid_experiment(run_duration_s=10.0, label="windup fixed")

In [ ]:
# ── Wind-up comparison plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col, (tof_x, pid_x, label) in enumerate([
    (tof_wu, pid_wu, 'Without Clamp (high KI)'),
    (tof_wf, pid_wf, 'With Clamp (KI=0.003, clamp ±1000)'),
]):
    real = tof_x[~tof_x['extrap']]

    # TOF row
    axes[0, col].plot(real['time_s'], real['dist_mm'], lw=1.5,
                      color='steelblue' if col == 0 else 'darkorange')
    axes[0, col].axhline(SETPOINT_MM, color='r', linestyle='--', lw=1.5)
    axes[0, col].set_ylabel('Distance (mm)')
    axes[0, col].set_title(f'{label}\nTOF Distance')
    axes[0, col].grid(alpha=0.3)

    # Motor row
    axes[1, col].plot(pid_x['time_s'], pid_x['motor_pwm'], 'm-', lw=1.5)
    axes[1, col].axhline(0, color='k', lw=0.5)
    axes[1, col].set_ylabel('Motor PWM')
    axes[1, col].set_xlabel('Time (s)')
    axes[1, col].set_title('Motor Output')
    axes[1, col].grid(alpha=0.3)

plt.suptitle('Integral Wind-up: Effect of Clamping', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lab5_windup_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -- PID control: 3 trials at max PWM ~ 40 / 80 / 120 -----------------------
KI_KP_RATIO   = 0.003 / 0.055
KP_PID_TRIALS = [0.005,  0.029,  0.059 ]
KI_PID_TRIALS = [round(kp * KI_KP_RATIO, 4) for kp in KP_PID_TRIALS]
KD_PID_TRIALS = [round(kp * KD_KP_RATIO, 4) for kp in KP_PID_TRIALS]

pid_trials = []
for pwm_t, kp_t, ki_t, kd_t in zip(PWM_TARGETS, KP_PID_TRIALS, KI_PID_TRIALS, KD_PID_TRIALS):
    set_pid_gains(kp=kp_t, ki=ki_t, kd=kd_t, setpoint=SETPOINT_MM)
    input(f'[PID PWM~{pwm_t}] Place robot 2-4 m from wall, press Enter ...')
    tof_t, pid_t = run_pid_experiment(run_duration_s=10.0, label=f'PID PWM~{pwm_t}')
    pid_trials.append((tof_t, pid_t, pwm_t))

set_pid_gains(kp=KP, ki=KI, kd=KD, setpoint=SETPOINT_MM)

# -- Plot --------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'darkorange', 'forestgreen']

ax = axes[0]
for (tof_t, _, pwm_t), color in zip(pid_trials, colors):
    real = tof_t[~tof_t['extrap']]
    ax.plot(real['time_s'], real['dist_mm'], color=color, lw=1.5, label=f'max PWM~{pwm_t}')
ax.axhline(SETPOINT_MM, color='r', linestyle='--', lw=1.5, label=f'Setpoint {SETPOINT_MM}mm')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Distance (mm)')
ax.set_title('PID Control - TOF Distance')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
for (tof_t, pid_t, pwm_t), kp_t, ki_t, kd_t, color in zip(pid_trials, KP_PID_TRIALS, KI_PID_TRIALS, KD_PID_TRIALS, colors):
    if len(pid_t):
        ax.plot(pid_t['time_s'], pid_t['motor_pwm'], color=color, lw=1.5, label=f'max PWM~{pwm_t}')
    elif len(tof_t):
        t_e, pwm_e = _estimate_motor_from_tof(tof_t, kp=kp_t, ki=ki_t, kd=kd_t)
        ax.plot(t_e, pwm_e, color=color, lw=1.5, linestyle='--', label=f'max PWM~{pwm_t} (est.)')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Motor PWM')
ax.set_title('PID Control - Motor Output')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('PID Control: max PWM 40 / 80 / 120', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lab5_pid_pwm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Extrapolation Validation

The TOF sensor returns new data every ~30–40 ms; the PID loop runs every ~1–2 ms.  
Without extrapolation the derivative term is 0 most iterations (stale error).  
With linear extrapolation the PID always sees a smoothly updated distance estimate.

This section:
1. Compares TOF update rate vs PID loop rate
2. Plots real vs extrapolated distance on the same graph
3. Shows the improvement in derivative signal quality

In [ ]:
# Use the last PID run data (tof_pid / pid_pid)
# If you want a fresh run, uncomment below:
# input("Place robot 2–4 m from wall, press Enter ...")
# tof_pid, pid_pid = run_pid_experiment(run_duration_s=10.0, label="extrap analysis")

if len(tof_pid) > 1 and len(pid_pid) > 1:
    real_only = tof_pid[~tof_pid['extrap']]
    all_tof   = tof_pid

    tof_rate = 1000.0 / real_only['time_ms'].diff().dropna().mean()
    pid_rate = 1000.0 / pid_pid['time_ms'].diff().dropna().mean()

    print(f"Real TOF rate  : {tof_rate:.1f} Hz  ({1000/tof_rate:.1f} ms per sample)")
    print(f"PID loop rate  : {pid_rate:.1f} Hz  ({1000/pid_rate:.1f} ms per sample)")
    print(f"Speed-up factor: {pid_rate/tof_rate:.1f}× faster than sensor")

In [ ]:
# ── Raw vs extrapolated overlay ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

real_tof   = tof_pid[~tof_pid['extrap']]
extrap_tof = tof_pid[ tof_pid['extrap']]

# Top: distance
ax = axes[0]
ax.plot(extrap_tof['time_s'], extrap_tof['dist_mm'], 'c.',
        ms=2, alpha=0.5, label='Extrapolated estimate')
ax.plot(real_tof['time_s'],   real_tof['dist_mm'],   'b.',
        ms=8, zorder=5,        label='Real TOF reading')
ax.axhline(SETPOINT_MM, color='r', linestyle='--', lw=1.5, label='Setpoint')
ax.set_ylabel('Distance (mm)')
ax.set_title('Real TOF vs Linear Extrapolation')
ax.legend()
ax.grid(alpha=0.3)

# Bottom: timestamps to illustrate rate difference
ax = axes[1]
if len(real_tof) > 1:
    intervals_real   = real_tof['time_ms'].diff().dropna()
    ax.plot(real_tof['time_s'].iloc[1:], intervals_real,
            'b-', lw=1.5, label='Real TOF interval (ms)')
if len(pid_pid) > 1:
    intervals_pid = pid_pid['time_ms'].diff().dropna()
    ax.plot(pid_pid['time_s'].iloc[1:], intervals_pid,
            'm-', lw=1, alpha=0.7, label='PID loop interval (ms)')
ax.set_ylabel('Interval (ms)')
ax.set_xlabel('Time (s)')
ax.set_title('Sampling Interval: TOF vs PID Loop')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('lab5_extrapolation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary = {
    'Controller': ['P only', 'PD', 'PID'],
    'KP': [0.015, 0.015, 0.015],
    'KI': [0.000, 0.000, 0.001],
    'KD': [0.000, 0.004, 0.004],
}

final_errors = []
for tof_x in [tof_p, tof_pd, tof_pid]:
    real = tof_x[~tof_x['extrap']]
    if len(real):
        last_dist = real['dist_mm'].iloc[-1]
        final_errors.append(int(last_dist - SETPOINT_MM))
    else:
        final_errors.append(None)

summary['Final Error (mm)'] = final_errors
print(pd.DataFrame(summary).to_string(index=False))

---
## 8. Disconnect

In [ ]:
ble.disconnect()
print("Disconnected from Artemis.")